In [1]:
%pip install pandas numpy openpyxl google-cloud-bigquery
%pip install --upgrade pyarrow db-dtypes

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: C:\Users\KrypRosa\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: C:\Users\KrypRosa\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

In [3]:
import pandas as pd

df_orders = pd.read_csv('Data/Raw/olist_orders_dataset.csv')
df_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
colunas_data = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in colunas_data:
    df_orders[col] = pd.to_datetime(df_orders[col], errors='coerce')


In [5]:
# Filtrar pedidos onde a data de entrega é nula e contar os status
pedidos_sem_entrega = df_orders[df_orders['order_delivered_customer_date'].isnull()]
pedidos_sem_entrega['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [6]:
# Carregar a tabela de produtos
df_products = pd.read_csv('Data/Raw/olist_products_dataset.csv')

print("--- Dados Nulos por Coluna em Produtos ---")
print(df_products.isna().sum())

--- Dados Nulos por Coluna em Produtos ---
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [7]:
# 1. Tratar os valores nulos da tabela de produtos
df_products['product_category_name'] = df_products['product_category_name'].fillna('nao_informado')

# Preencher os numéricos nulos com 0
colunas_numericas = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
df_products[colunas_numericas] = df_products[colunas_numericas].fillna(0)

# 2. Carregar a tabela de tradução para dar uma olhada nela
df_translation = pd.read_csv('Data/Raw/product_category_name_translation.csv')

print("--- Nulos após o tratamento em Produtos ---")
print(df_products.isna().sum())

print("\n--- Primeiras linhas da tabela de tradução ---")
print(df_translation.head())

--- Nulos após o tratamento em Produtos ---
product_id                      0
product_category_name           0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64

--- Primeiras linhas da tabela de tradução ---
    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


In [8]:
# 1. Fazer o Merge (Join) da tabela de produtos com a tabela de tradução
df_products_translated = pd.merge(
    df_products, 
    df_translation, 
    on='product_category_name', 
    how='left'
)

# 2. Tratar o 'nao_informado' na nova coluna em inglês que foi criada
df_products_translated['product_category_name_english'] = df_products_translated['product_category_name_english'].fillna('not_defined')

# 3. Conferir o resultado olhando os nulos da coluna em inglês e as primeiras linhas
print("--- Quantidade de nulos na coluna em inglês ---")
print(df_products_translated['product_category_name_english'].isna().sum())

print("\n--- Visualização das colunas de categoria ---")
df_products_translated[['product_category_name', 'product_category_name_english']].head()


--- Quantidade de nulos na coluna em inglês ---
0

--- Visualização das colunas de categoria ---


,product_category_name,product_category_name_english
0,perfumaria,perfumery
1,artes,art
2,esporte_lazer,sports_leisure
3,bebes,baby
4,utilidades_domesticas,housewares


In [9]:
# Carregar a tabela de geolocalização
df_geo = pd.read_csv('Data/Raw/olist_geolocation_dataset.csv')

print("--- Total de Linhas Originais ---")
print(len(df_geo))

print("\n--- Quantidade de CEPs Únicos ---")
print(df_geo['geolocation_zip_code_prefix'].nunique())

print("\n--- Linhas 100% Duplicadas ---")
print(df_geo.duplicated().sum())


--- Total de Linhas Originais ---
1000163

--- Quantidade de CEPs Únicos ---
19015

--- Linhas 100% Duplicadas ---
261831


In [10]:
# 1. Agrupar por CEP e calcular a média das coordenadas, pegando a primeira cidade/estado encontrada
df_geo_cleaned = df_geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

# 2. Conferir o novo tamanho da tabela limpa
print("--- Novo Total de Linhas da Tabela Limpa ---")
print(len(df_geo_cleaned))

# 3. Dar uma espiada no resultado
df_geo_cleaned.head()

--- Novo Total de Linhas da Tabela Limpa ---
19015


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


In [11]:
# Carregar a tabela de pagamentos
df_payments = pd.read_csv('Data/Raw/olist_order_payments_dataset.csv')

print("--- Resumo da Tabela de Pagamentos ---")
df_payments.info()

print("\n--- Estatísticas dos Valores e Parcelas ---")
print(df_payments[['payment_installments', 'payment_value']].describe())

print("\n--- Quantidade de Pagamentos com Valor Zerado ou Negativo ---")
print((df_payments['payment_value'] <= 0).sum())

--- Resumo da Tabela de Pagamentos ---
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB

--- Estatísticas dos Valores e Parcelas ---
       payment_installments  payment_value
count         103886.000000  103886.000000
mean               2.853349     154.100380
std                2.687051     217.494064
min                0.000000       0.000000
25%                1.000000      56.790000
50%                1.000000     100.000000
75%                4.000000     171.837500
max               24.000000   13664.080000

--- Qua

In [12]:
# 1. Investigar quais tipos de pagamento geraram valor zerado
print("--- Tipos de pagamento dos registros zerados ---")
print(df_payments[df_payments['payment_value'] == 0]['payment_type'].value_counts())

# 2. Corrigir as parcelas zeradas: se for 0, transformamos em 1 (à vista)
df_payments['payment_installments'] = df_payments['payment_installments'].replace(0, 1)

# 3. Validar se a correção das parcelas funcionou (o mínimo deve virar 1)
print("\n--- Novo valor mínimo de parcelas ---")
print(df_payments['payment_installments'].min())

--- Tipos de pagamento dos registros zerados ---
payment_type
voucher        6
not_defined    3
Name: count, dtype: int64

--- Novo valor mínimo de parcelas ---
1


In [13]:
traducoes_faltantes = {
    'pc_gamer': 'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos': 'portable_kitchen_food_preparers'
}

df_products_translated['product_category_name_english'] = (
    df_products_translated['product_category_name_english']
    .fillna(df_products_translated['product_category_name'].map(traducoes_faltantes))
    .fillna('not_defined')
)

In [14]:
df_geo_br = df_geo[
    df_geo['geolocation_lat'].between(-34, 6) &
    df_geo['geolocation_lng'].between(-74, -34)
]

df_geo_cleaned = df_geo_br.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'median',
    'geolocation_lng': 'median',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

In [15]:
colunas_num = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
df_products[colunas_num] = df_products[colunas_num].fillna(0)